# 02 — State A Dataset Original

Load the fixed dataset sample, inspect the original prompt, execute the untouched CAD code, and visualize the baseline geometry.


In [ ]:
from base64 import b64encode
from pathlib import Path
import json
import logging
import os
import sys

import yaml
from IPython.display import HTML, Image, Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "code_base").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
MODULE_ROOT = PROJECT_ROOT / "code_base" / "fea_cad_one_sample"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

API_ENV_PATH = MODULE_ROOT / "src" / "api.env"


def _load_api_env(path: Path) -> dict[str, str]:
    env_values: dict[str, str] = {}
    if not path.exists():
        return env_values
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip("'").strip('"')
        if key and value:
            env_values[key] = value
            os.environ.setdefault(key, value)
    return env_values


api_env_values = _load_api_env(API_ENV_PATH)

from src import interfaces as api

logging.basicConfig(level=logging.INFO, format="%(name)s | %(levelname)s | %(message)s")

FIXED_SAMPLE_PATH = MODULE_ROOT / "experiment_config" / "fixed_sample.yaml"
OUTPUT_ROOT = MODULE_ROOT / "outputs"
sample_config = yaml.safe_load(FIXED_SAMPLE_PATH.read_text(encoding="utf-8"))
sample_id = str(sample_config["sample_id"])
selection_source = str(sample_config.get("load_source", "dataset"))
connection_string = os.environ.get("CAD_DB_CONNECTION_STRING")
state_a_dir = OUTPUT_ROOT / f"sample_{sample_id}" / "01_dataset_original"
state_a_dir.mkdir(parents=True, exist_ok=True)
pipeline_config = api.PipelineConfig(
    config_name="config_gpt_5_4_mini.yaml",
    config_path=MODULE_ROOT / "src" / "copied_from_cadcodeverify" / "configs" / "config_gpt_5_4_mini.yaml",
    output_root=OUTPUT_ROOT,
    force=True,
)

print("[STAGE] Setup")
print(f"  → sample id      : {sample_id}")
print(f"  → selection mode : {selection_source}")
print(f"  → state A dir    : {state_a_dir}")
print(f"  → output root    : {OUTPUT_ROOT}")
print(f"  → api.env        : {API_ENV_PATH}")
print(f"  → api.env keys   : {sorted(api_env_values)}")

In [ ]:
print("[STAGE] Load sample")
sample = api.load_selected_sample(
    module_root=MODULE_ROOT,
    sample_id=sample_id,
    selection_source=selection_source,
    connection_string=connection_string,
)
assert sample.sample_id == sample_id
assert bool(sample.prompt.strip())
assert bool(sample.ground_truth_code.strip())

display(Markdown(f"## Original {selection_source} prompt"))
display(Markdown("```text\n" + sample.prompt + "\n```"))
print(f"  → prompt variant : {sample.prompt_variant}")
print(f"  → code length    : {len(sample.ground_truth_code)}")
print(f"  → prompt hash    : {__import__('hashlib').sha256(sample.prompt.encode('utf-8')).hexdigest()}")
print(f"  → source         : {sample.source}")
print("  ✓ sample loaded and prompt displayed")

In [ ]:
print("[STAGE] Execute original DB code and render views")
original_code = api.generate_original_code(sample, pipeline_config)
assert original_code == sample.ground_truth_code
execution = api.execute_and_export_cadquery(original_code, state_a_dir, basename="original", force=True)
view_paths = api.render_views(Path(execution["stl_path"]), state_a_dir / "views", force=True)
metrics_json_path = state_a_dir / "geometry_metrics.json"
metrics_md_path = state_a_dir / "geometry_metrics.md"
metrics = api.compute_geometry_metrics({"State A": Path(execution["stl_path"])}, metrics_json_path, force=True)
api.build_geometry_metrics_markdown(metrics, metrics_md_path, force=True)

prompt_path = state_a_dir / "original_prompt.txt"
code_path = state_a_dir / "database_original_code.py"
metadata_path = state_a_dir / "metadata.json"
provenance_path = state_a_dir / "provenance.json"
step_path = Path(execution["step_path"])
stl_path = Path(execution["stl_path"])
execution_log_path = Path(execution["execution_log_path"])

print(f"  → prompt file : {prompt_path}")
print(f"  → code file   : {code_path}")
print(f"  → step file   : {step_path}")
print(f"  → stl file    : {stl_path}")
print(f"  → log file    : {execution_log_path}")
print(f"  → views       : {sorted(view_paths.keys())}")
print(f"  → metrics     : {metrics_json_path}")
print("")
display(Markdown("## Shape views"))
for name in ["front", "side", "top", "iso", "grid"]:
    display(Markdown(f"**{name}**"))
    display(Image(filename=view_paths[name]))

display(Markdown("## Geometry metrics"))
display(Markdown(metrics_md_path.read_text(encoding="utf-8")))


In [ ]:
print("[STAGE] Assertions")
assert prompt_path.exists()
assert code_path.exists()
assert metadata_path.exists()
assert provenance_path.exists()
assert step_path.exists()
assert stl_path.exists()
assert execution_log_path.exists()
assert all(Path(path).exists() for path in view_paths.values())
assert metrics_json_path.exists()
assert metrics_md_path.exists()
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
provenance = json.loads(provenance_path.read_text(encoding="utf-8"))
print(f"  → metadata keys  : {sorted(metadata.keys())}")
print(f"  → provenance keys: {sorted(provenance.keys())}")
print(f"  → execution log  : {execution_log_path}")
assert metadata["sample_id"] == sample_id
assert provenance["sample_id"] == sample_id
assert provenance["prompt_source"] == "database"
print("  ✓ State A baseline is saved and reproducible")


## What this notebook proves

- The original dataset CAD code executes unchanged.
- The baseline STEP/STL and views are produced.
- The baseline geometry is measurable.
